In [2]:
!python -u lab07_starter.py

train (1438, 64) test (359, 64)
autoencoder test reconstruction MSE: 0.0163
feature network accuracy on real test images: 0.972
autoencoder samples from N(0, I): {'frechet': 86.35, 'precision': 0.435, 'recall': 0.772, 'nearest_train': 2.028}
saved starter_check.png


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab07_starter as S

Xtr, ytr, Xte, yte = S.load_task()
print("Train:", Xtr.shape, "Test:", Xte.shape)

Train: torch.Size([1438, 64]) Test: torch.Size([359, 64])


# 1 - VAE

In [16]:
class VAE(nn.Module):
    def __init__(self, d_in=64, d_hidden=64, d_latent=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU())
        self.mu_layer = nn.Linear(d_hidden, d_latent)
        self.logvar_layer = nn.Linear(d_hidden, d_latent)
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_in)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_layer(h), self.logvar_layer(h)

    def reparameterize(self, mu, logvar):
        eps = torch.randn_like(mu)
        return mu + torch.exp(0.5 * logvar) * eps

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [17]:
def kl_per_image(mu, logvar):
    """KL[q(z|x) || N(0,I)]: sum over the eight latent dimensions."""
    return -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(dim=1)


def reconstruction_per_image(logits, x):
    """Bernoulli negative log likelihood: sum over all 64 pixels."""
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=1)


def vae_loss(model, xb, yb=None, beta=1.0):
    """Mean over images of [sum_pixel BCE + beta * sum_latent KL]."""
    logits, mu, logvar = model(xb)
    return (reconstruction_per_image(logits, xb) + beta * kl_per_image(mu, logvar)).mean()

In [18]:
# Check 1: Closed-form KL vs Monte Carlo KL
torch.manual_seed(123)

mu = torch.randn(5, 8) * 0.5
logvar = torch.randn(5, 8) * 0.5

eps = torch.randn(50000, 5, 8)
z = mu.unsqueeze(0) + torch.exp(0.5 * logvar).unsqueeze(0) * eps

mc_kl = 0.5 * (
    z**2 - eps**2 - logvar.unsqueeze(0)
).sum(dim=2).mean(dim=0)

exact_kl = kl_per_image(mu, logvar)

print("KL difference:", (mc_kl - exact_kl).abs().max().item())
assert torch.allclose(mc_kl, exact_kl, atol=0.05, rtol=0)

KL difference: 0.02042853832244873


In [19]:
# Check 2: Gradient with and without reparameterization
torch.manual_seed(0)
model = VAE()
x = Xtr[:32]

logits, mu, logvar = model(x)
recon = reconstruction_per_image(logits, x).mean()

grad = torch.autograd.grad(recon, model.mu_layer.weight)[0]
with_trick = grad.norm().item()

z = model.reparameterize(mu, logvar).detach()
recon_detached = reconstruction_per_image(model.decode(z), x).mean()

grad_detached = torch.autograd.grad(
    recon_detached,
    model.mu_layer.weight,
    allow_unused=True
)[0]

without_trick = 0.0 if grad_detached is None else grad_detached.norm().item()

print("Gradient with trick:", with_trick)
print("Gradient without trick:", without_trick)

assert with_trick > 0
assert without_trick == 0.0

Gradient with trick: 0.8507521152496338
Gradient without trick: 0.0


In [20]:
# Check 3: Beta = 1 equals the negative ELBO
logits, mu, logvar = model(x)

recon = reconstruction_per_image(logits, x)
kl = kl_per_image(mu, logvar)

loss = (recon + kl).mean()

log_px = (
    x * F.logsigmoid(logits)
    + (1 - x) * F.logsigmoid(-logits)
).sum(dim=1)

negative_elbo = -(log_px - kl).mean()

print("VAE loss:", loss.item())
print("Negative ELBO:", negative_elbo.item())

assert torch.allclose(loss, negative_elbo, atol=1e-5)

VAE loss: 44.81682586669922
Negative ELBO: 44.81682586669922


# 2 - Autoencoder vs VAE

In [21]:
SEEDS = (0, 1, 2)
BETAS = (0.1, 0.5, 1.0, 2.0, 4.0)

EPOCHS = 200
LR = 2e-3
BATCH_SIZE = 64
D_LATENT = 8
D_HIDDEN = 64
N_SAMPLES = len(Xte)

net = S.train_feature_net(Xtr, ytr, seed=0)
net.eval()
fte = S.get_features(net, Xte)

def draw_prior(seed, n):
    g = torch.Generator().manual_seed(1000 + seed)
    return torch.randn(n, D_LATENT, generator=g)


@torch.no_grad()
def vae_samples(model, z):
    return model.decode(z).sigmoid()


@torch.no_grad()
def vae_test_mse(model):
    mu, _ = model.encode(Xte)
    return ((model.decode(mu).sigmoid() - Xte)**2).mean().item()

In [22]:
ae_models = {}
vae_models = {}
results = []
spreads = []

for seed in SEEDS:

    torch.manual_seed(seed)
    ae = S.AE(d_hidden=D_HIDDEN, d_latent=D_LATENT)
    S.train(
        ae, S.ae_loss, Xtr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=seed
    )
    ae.eval()
    ae_models[seed] = ae

    torch.manual_seed(seed)
    vae = VAE(d_hidden=D_HIDDEN, d_latent=D_LATENT)
    S.train(
        vae, vae_loss, Xtr,
        epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=seed
    )
    vae.eval()
    vae_models[seed] = vae

    z = draw_prior(seed, N_SAMPLES)

    with torch.no_grad():
        ae_generated = ae.decode(z)
        vae_generated = vae_samples(vae, z)

        ae_codes = ae.encode(Xtr)
        vae_codes = vae.encode(Xtr)[0]

    for name, samples, mse, codes in [
        ("AE", ae_generated, S.reconstruction_mse(ae, Xte), ae_codes),
        ("VAE", vae_generated, vae_test_mse(vae), vae_codes)
    ]:
        scores = S.score_samples(samples, fte, net, Xtr)

        results.append({
            "seed": seed,
            "model": name,
            "test_mse": mse,
            **scores
        })

        stds = codes.std(dim=0, unbiased=False).tolist()

        spreads.append({
            "seed": seed,
            "model": name,
            **{f"dim_{i}": std for i, std in enumerate(stds)}
        })

    S.save_grid(
        ae_generated[:30],
        f"part2_ae_seed{seed}.png",
        title=f"AE seed {seed}"
    )

    S.save_grid(
        vae_generated[:30],
        f"part2_vae_seed{seed}.png",
        title=f"VAE seed {seed}"
    )

    print("Finished seed", seed)

Finished seed 0
Finished seed 1
Finished seed 2


In [23]:
part2_df = pd.DataFrame(results)
spread_df = pd.DataFrame(spreads)

print("Reconstruction and sample quality:")
display(part2_df.round(4))

print("Latent code standard deviation:")
display(spread_df.round(4))

Reconstruction and sample quality:


,seed,model,test_mse,frechet,precision,recall,nearest_train
0,0,AE,0.0152,70.8812,0.3510,0.8663,2.0602
1,0,VAE,0.0259,12.9154,0.9136,0.4150,1.1303
2,1,AE,0.0151,87.3034,0.3593,0.7660,2.0956
3,1,VAE,0.0265,11.3869,0.8997,0.4680,1.1429
4,2,AE,0.0149,110.7445,0.4290,0.6574,2.3304
5,2,VAE,0.0263,13.4287,0.8830,0.4875,1.1366


Latent code standard deviation:


,seed,model,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7
0,0,AE,0.6831,0.8677,0.5646,0.6289,0.6887,0.8061,0.5306,0.6687
1,0,VAE,0.9400,0.9608,0.9366,0.4813,0.0216,0.8235,0.0206,0.0340
2,1,AE,0.6299,0.5758,0.7564,0.6445,1.0876,0.6603,0.7445,0.6449
3,1,VAE,0.9037,0.4447,0.8114,0.9113,0.8938,0.0214,0.0249,0.0377
4,2,AE,0.7149,0.7585,0.7503,0.7040,0.6167,0.7058,0.7707,0.7262
5,2,VAE,0.8947,0.9128,0.9039,0.0263,0.5364,0.8178,0.0302,0.0217
